# Alpamayo 2 — 実 VLM バックボーンでの GPU スモークテスト (Kaggle Tesla T4)

> [alpamayo2.md](alpamayo2.md) / [alpamayo2.ipynb](alpamayo2.ipynb) の GPU leg。
> CPU 版が自作のミニ VLM だったのに対し、こちらは **実在の Vision-Language モデルを実 GPU にロード**して
> Alpamayo と同じパイプライン（VLM prefill → CoC 推論テキスト → hidden state を条件に flow matching → 軌道）を回す。

## なぜ Alpamayo 本体を使わないのか

| モデル | パラメータ | 必要 VRAM | T4 (16GB) |
|---|---|---|---|
| `nvidia/Alpamayo2-Super` | 34B | H100 80GB × 2 相当（実測 67 / 71 GiB） | 不可 |
| `nvidia/Alpamayo-1.5-10B` | 10.5B | 24GB 以上（重み 22GB） | 不可 |
| **`Qwen/Qwen3-VL-2B-Instruct`** | **2.13B** | **約 4.3GB (fp16)** | **可** |

Alpamayo 2 Super の `config.json` は `vlm_config.model_type = "qwen3_vl"` である。
つまり**バックボーンは Qwen3-VL アーキテクチャ**であり、その最小メンバーである 2B を代替に選ぶことで、
同じ系統のモデルで同じコードパスを踏める。`Qwen3-VL-8B-Instruct` は重みだけで約 17.5GB あり T4 には載らない。

**action expert 側は代用ではない。** `PerWaypointActionInProjV2` / `FlowMatching` /
`UnicycleAccelCurvatureActionSpace` は公式実装の移植で、公式 `config.json` の実数値を使う。
それを**実 VLM の hidden state を条件として GPU 上で実際に学習させ**、収束するかを assert で検証する。

すべてのチェックポイントは `assert` で書かれているので、最後まで緑で通ったこと自体が検証結果である。


---

### Provenance

この notebook の出力は、実際に Kaggle の **NVIDIA Tesla T4** 上で実行した結果を 実行ログから焼き戻したものである（貼り付けや手書きではない）。

- kernel: `morimori040506/alpamayo2-gpu-smoke`
- GPU: Tesla T4 (compute capability 7.5), 14.6 GiB
- 環境: Python 3.12 / torch 2.10.0+cu128 / transformers 5.0.0 / CUDA 12.8
- 実行日: 2026-08-07 (UTC)

再現コマンド:

```bash
kaggle kernels push -p . --accelerator NvidiaTeslaT4
kaggle kernels status morimori040506/alpamayo2-gpu-smoke
kaggle kernels output morimori040506/alpamayo2-gpu-smoke -p out/
```

**注意**: 焼き戻せるのは stdout のみである。matplotlib の図は実行ログに含まれないため、図を出すセルの出力は空になっている。図を見るには上記のコマンドで自分で実行すること。


## 1. 環境の確認とハードガード

T4 は compute capability 7.5 (Turing) である。ここで 2 つの罠を潰しておく。

- `torch.cuda.is_bf16_supported()` は Ampere 以前でもソフトウェアエミュレーションで `True` を返す。
  dtype 判定は必ず `torch.cuda.get_device_capability() >= (8, 0)` で行う。
- FlashAttention-2 は Ampere 以降 (sm_80+) のみ。Turing では `attn_implementation="sdpa"` を明示する。


In [1]:
import subprocess, sys

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
+-----------------------------------------+------------------------+----------------------+
|   1  Tesla T4                       Off |   00000000:00:05.0 Off |                    0 |
| N/A   47C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
+-----------------------------------------+------------------------+------------

In [2]:
import math, os, time, json, gc

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import transformers
from PIL import Image

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda        :", torch.version.cuda)

assert torch.cuda.is_available(), "GPU が見えていない。Kaggle の Accelerator 設定を確認すること"

DEVICE = "cuda"
cap = torch.cuda.get_device_capability()
gpu_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"\nGPU         : {gpu_name}")
print(f"capability  : sm_{cap[0]}{cap[1]}")
print(f"total VRAM  : {total_vram:.1f} GiB")

# --- ハードガード 1: 古すぎる GPU を掴んでいないか
assert cap >= (7, 0), f"compute capability {cap} は非対応 (sm_70 以上が必要)"

# --- ハードガード 2: bf16 判定は必ず capability で行う
BF16_OK = cap >= (8, 0)
DTYPE = torch.bfloat16 if BF16_OK else torch.float16
print(f"\ntorch.cuda.is_bf16_supported() = {torch.cuda.is_bf16_supported()}  <- これは信用しない")
print(f"get_device_capability() >= (8,0) = {BF16_OK}          <- これで判定する")
print(f"=> 選択した dtype: {DTYPE}")

# --- ハードガード 3: flash-attn は Turing 非対応なので sdpa を使う
ATTN_IMPL = "flash_attention_2" if cap >= (8, 0) else "sdpa"
print(f"=> attn_implementation: {ATTN_IMPL}")

if not BF16_OK:
    assert DTYPE == torch.float16, "sm_80 未満で bf16 を選ぶと数値が壊れる"
    print("\n[NOTE] T4 は bf16 非対応のため fp16 で走る。fp16 は数値不安定になりうるので")
    print("       このあと hidden state の NaN チェックを入れ、NaN が出たら fp32 に落とす。")

torch.manual_seed(0)
np.random.seed(0)
print("\nCHECKPOINT 1 OK: environment guards passed")


python      : 3.12.13
torch       : 2.10.0+cu128
transformers: 5.0.0
cuda        : 12.8

GPU         : Tesla T4
capability  : sm_75
total VRAM  : 14.6 GiB

torch.cuda.is_bf16_supported() = True  <- これは信用しない
get_device_capability() >= (8,0) = False          <- これで判定する
=> 選択した dtype: torch.float16
=> attn_implementation: sdpa

[NOTE] T4 は bf16 非対応のため fp16 で走る。fp16 は数値不安定になりうるので
       このあと hidden state の NaN チェックを入れ、NaN が出たら fp32 に落とす。

CHECKPOINT 1 OK: environment guards passed


## 2. 公式 config の実数値と、公式実装の移植

`nvidia/Alpamayo2-Super` の `config.json` からそのまま持ってきた値を使う。
以下のクラスは `NVlabs/alpamayo2` の移植である（CPU 版 notebook と同一のコード）。


In [3]:
# nvidia/Alpamayo2-Super : config.json の実数値
N_WAYPOINTS = 64
DT = 0.1
ACCEL_BOUNDS = (-9.8, 9.8)
CURVATURE_BOUNDS = (-0.33, 0.33)
ACCEL_MEAN = 0.02902694707164455
ACCEL_STD = 0.6810426736454882
CURVATURE_MEAN = 0.0002692167976330542
CURVATURE_STD = 0.026148280660833106
NUM_INFERENCE_STEPS = 10
LOW_SPEED_CURVATURE_THRESHOLD_MPS = 0.6


class UnicycleAccelCurvatureActionSpace(nn.Module):
    """NVlabs/alpamayo2 : action_space/unicycle_accel_curvature.py の移植 (action->traj)."""

    def __init__(self):
        super().__init__()
        self.dt, self.n_waypoints = DT, N_WAYPOINTS
        self.accel_bounds, self.curvature_bounds = ACCEL_BOUNDS, CURVATURE_BOUNDS

    def get_action_space_dims(self):
        return (self.n_waypoints, 2)

    def is_within_bounds(self, action):
        accel = action[..., 0] * ACCEL_STD + ACCEL_MEAN
        kappa = action[..., 1] * CURVATURE_STD + CURVATURE_MEAN
        ok_a = (accel >= self.accel_bounds[0]) & (accel <= self.accel_bounds[1])
        ok_k = (kappa >= self.curvature_bounds[0]) & (kappa <= self.curvature_bounds[1])
        return torch.all(ok_a & ok_k, dim=-1)

    def action_to_traj(self, action, v0):
        accel = action[..., 0] * ACCEL_STD + ACCEL_MEAN
        kappa = action[..., 1] * CURVATURE_STD + CURVATURE_MEAN
        dt, dt_2_term, half_dt = self.dt, 0.5 * self.dt ** 2, 0.5 * self.dt
        velocity = torch.cat([v0.unsqueeze(-1),
                              v0.unsqueeze(-1) + torch.cumsum(accel * dt, dim=-1)], dim=-1)
        initial_yaw = torch.zeros_like(v0)
        theta = torch.cat([initial_yaw.unsqueeze(-1),
                           initial_yaw.unsqueeze(-1)
                           + torch.cumsum(kappa * velocity[..., :-1] * dt, dim=-1)
                           + torch.cumsum(kappa * accel * dt_2_term, dim=-1)], dim=-1)
        x = (torch.cumsum(velocity[..., :-1] * torch.cos(theta[..., :-1]) * half_dt, dim=-1)
             + torch.cumsum(velocity[..., 1:] * torch.cos(theta[..., 1:]) * half_dt, dim=-1))
        y = (torch.cumsum(velocity[..., :-1] * torch.sin(theta[..., :-1]) * half_dt, dim=-1)
             + torch.cumsum(velocity[..., 1:] * torch.sin(theta[..., 1:]) * half_dt, dim=-1))
        return torch.stack([x, y], dim=-1), velocity, theta


class FlowMatching:
    """NVlabs/alpamayo2 : diffusion/flow_matching.py の移植."""

    def __init__(self, x_dims, num_inference_steps=NUM_INFERENCE_STEPS):
        self.x_dims = x_dims
        self.num_inference_steps = num_inference_steps
        self.beta_dist = torch.distributions.beta.Beta(torch.tensor(1.5), torch.tensor(1.0))
        self.beta_scale_constant = 0.999

    def sample_timesteps(self, batch_size, device):
        t = self.beta_dist.sample((batch_size,)).to(device)
        return self.beta_scale_constant - t * self.beta_scale_constant

    def construct_training_data(self, x):
        t = self.sample_timesteps(x.shape[0], x.device)
        while len(t.shape) < len(x.shape):
            t = t.unsqueeze(-1)
        noise = torch.randn_like(x)
        return {"x": x, "noisy_x": t * x + (1 - t) * noise, "timesteps": t, "noise": noise}

    @staticmethod
    def compute_loss_from_pred(training_data, pred):
        target = (training_data["x"] - training_data["noise"]).to(dtype=pred.dtype)
        return F.mse_loss(target, pred)

    @torch.no_grad()
    def sample(self, batch_size, step_fn, device, inference_step=None):
        inference_step = inference_step or self.num_inference_steps
        x = torch.randn(batch_size, *self.x_dims, device=device)
        time_steps = torch.linspace(0.0, 1.0, inference_step + 1, device=device)
        n_dim = len(self.x_dims)
        for i in range(inference_step):
            dt = (time_steps[i + 1] - time_steps[i]).view(1, *[1] * n_dim).expand(batch_size, *[1] * n_dim)
            t_start = time_steps[i].view(1, *[1] * n_dim).expand(batch_size, *[1] * n_dim)
            x = x + dt * step_fn(x=x, t=t_start)
        return x


class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        out = x.float() * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return out.type_as(x) * self.weight


class FourierEncoderV2(nn.Module):
    def __init__(self, dim, max_freq=100.0):
        super().__init__()
        half = dim // 2
        self.out_dim = dim
        self.register_buffer("freqs", torch.logspace(0, math.log10(max_freq), steps=half)[None, :])

    def forward(self, x):
        arg = x[..., None] * self.freqs * 2 * torch.pi
        return torch.cat([torch.sin(arg), torch.cos(arg)], -1) * math.sqrt(2)


class MLPEncoder(nn.Module):
    def __init__(self, num_input_feats, num_enc_layers, hidden_size, outdim):
        super().__init__()
        layers = [nn.Linear(num_input_feats, hidden_size), nn.SiLU()]
        for i in range(num_enc_layers):
            if i < num_enc_layers - 1:
                layers += [RMSNorm(hidden_size, 1e-5), nn.Linear(hidden_size, hidden_size), nn.SiLU()]
            else:
                layers += [RMSNorm(hidden_size, 1e-5), nn.Linear(hidden_size, outdim)]
        self.trunk = nn.Sequential(*layers)

    def forward(self, x):
        return self.trunk(x)


class PerWaypointActionInProjV2(nn.Module):
    """NVlabs/alpamayo2 : models/action_in_proj.py の移植."""

    def __init__(self, in_dims, out_dim, num_enc_layers=4, hidden_size=256,
                 max_freq=100.0, num_fourier_feats=20):
        super().__init__()
        self.sinus = nn.ModuleList(FourierEncoderV2(num_fourier_feats, max_freq)
                                   for _ in range(in_dims[-1]))
        self.timestep_fourier_encoder = FourierEncoderV2(num_fourier_feats, max_freq)
        n_in = sum(e.out_dim for e in self.sinus) + self.timestep_fourier_encoder.out_dim
        self.encoder = MLPEncoder(n_in, num_enc_layers, hidden_size, out_dim)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x, timesteps):
        b, n, _ = x.shape
        x, timesteps = x.float(), timesteps.float()
        action_feats = torch.cat([e(x[:, :, i]) for i, e in enumerate(self.sinus)], dim=-1)
        ts = self.timestep_fourier_encoder(timesteps[..., -1]).repeat(1, n, 1)
        feats = torch.cat((action_feats, ts), dim=-1)
        return self.norm(self.encoder(feats.flatten(0, 1)).reshape(b, n, -1))


action_space = UnicycleAccelCurvatureActionSpace().to(DEVICE)
diffusion = FlowMatching(x_dims=action_space.get_action_space_dims())

# サニティチェック: a=0, kappa=0 なら 10 m/s で 6.4 秒 -> 64 m 直進
norm_zero = torch.zeros(1, N_WAYPOINTS, 2, device=DEVICE)
norm_zero[..., 0] = (0.0 - ACCEL_MEAN) / ACCEL_STD
norm_zero[..., 1] = (0.0 - CURVATURE_MEAN) / CURVATURE_STD
xy, _, _ = action_space.action_to_traj(norm_zero, torch.tensor([10.0], device=DEVICE))
print(f"unicycle 積分チェック: x(6.4s) = {xy[0,-1,0]:.4f} m (理論値 64.0), "
      f"y(6.4s) = {xy[0,-1,1]:.6f} m (理論値 0.0)")
assert abs(xy[0, -1, 0].item() - 64.0) < 1e-2
assert abs(xy[0, -1, 1].item()) < 1e-5
print("\nCHECKPOINT 2 OK: official action space / flow matching ported and verified")


unicycle 積分チェック: x(6.4s) = 64.0000 m (理論値 64.0), y(6.4s) = 0.000000 m (理論値 0.0)

CHECKPOINT 2 OK: official action space / flow matching ported and verified


## 3. 合成運転シーン（マルチカメラ RGB）と Chain-of-Causation ラベル

実車載データ (`nvidia/PhysicalAI-Autonomous-Vehicles`) は gated かつ大容量なので、
ここでは VLM が実際に「見て判断できる」合成シーンを RGB で描く。
CoC の driving decision は公式の閉集合語彙のサブセットを使う。


In [4]:
LON_DECISIONS = ["Set speed tracking", "Lead obstacle following", "Stop for static constraints"]
LAT_DECISIONS = ["Lane keeping & centering", "Lane change (lateral push)", "Turn (intersection)"]
CAM_NAMES = ["front_wide", "cross_left", "cross_right"]
IMG_SIZE = 112


def render_rgb_scene(lead_dist, light_red, lane_offset, turn_cmd, lane_change_cmd, rng):
    """3 カメラ分の RGB 画像 (PIL) を返す."""
    out = []
    S = IMG_SIZE
    # --- front_wide
    im = np.zeros((S, S, 3), dtype=np.float32)
    im[:S // 2] = np.array([0.45, 0.55, 0.75])          # 空
    im[S // 2:] = np.array([0.32, 0.32, 0.34])          # 路面
    for c in range(3):                                  # 車線
        lx = int(S / 2 - 26 + c * 26)
        im[S // 2:, max(lx - 1, 0):lx + 1] = np.array([0.85, 0.85, 0.80])
    if lead_dist < 60:                                  # 先行車 (近いほど大きい)
        sz = int(np.clip(46 - lead_dist * 0.55, 6, 44))
        cy, cx = int(S / 2 + sz * 0.35), S // 2
        y0, y1 = max(cy - sz // 2, 0), min(cy + sz // 2, S)
        x0, x1 = max(cx - sz // 2, 0), min(cx + sz // 2, S)
        im[y0:y1, x0:x1] = np.array([0.75, 0.15, 0.12])
        im[y1 - max(sz // 6, 2):y1, x0:x1] = np.array([0.95, 0.35, 0.10])   # テールランプ
    tl = np.array([0.95, 0.12, 0.12]) if light_red else np.array([0.15, 0.85, 0.30])
    im[6:16, S // 2 - 5:S // 2 + 5] = tl                # 信号
    out.append(im)

    # --- cross_left / cross_right
    for sign in (-1.0, +1.0):
        side = np.zeros((S, S, 3), dtype=np.float32)
        side[:S // 3] = np.array([0.45, 0.55, 0.75])
        side[S // 3:] = np.array([0.30, 0.30, 0.32])
        col = int(np.clip(S / 2 + sign * lane_offset * 14 + turn_cmd * 26
                          + lane_change_cmd * 18, 3, S - 4))
        side[S // 3:, col - 3:col + 3] = np.array([0.88, 0.88, 0.82])
        out.append(side)

    imgs = []
    for a in out:
        a = np.clip(a + rng.normal(0, 0.02, a.shape), 0, 1)
        imgs.append(Image.fromarray((a * 255).astype(np.uint8)))
    return imgs


def expert_driver(lead_dist, light_red, lane_offset, turn_cmd, lane_change_cmd, v0):
    t = np.arange(N_WAYPOINTS) * DT
    if light_red and lead_dist > 15:
        lon = "Stop for static constraints"
        accel = np.full(N_WAYPOINTS, -min(2.5, v0 / 3.0))
    elif lead_dist < 25:
        lon = "Lead obstacle following"
        accel = -1.6 * np.exp(-t / 3.0) * (25 - lead_dist) / 25 * 2.0
        v_prof = v0 + np.cumsum(accel * DT)
        if v_prof.min() < 1.5:
            accel *= (v0 - 1.5) / max(v0 - v_prof.min(), 1e-6)
    else:
        lon = "Set speed tracking"
        accel = 0.9 * np.exp(-t / 2.5)

    if turn_cmd != 0:
        lat = "Turn (intersection)"
        kappa = np.where(t > 1.2, 0.05 * np.sign(turn_cmd), 0.0)
    elif lane_change_cmd != 0:
        lat = "Lane change (lateral push)"
        kappa = 0.010 * np.sign(lane_change_cmd) * np.sin(2 * np.pi * t / 5.0) * (t < 5.0)
    else:
        lat = "Lane keeping & centering"
        kappa = -0.002 * lane_offset * np.sin(2 * np.pi * t / 3.0) * (t < 3.0)
    return lon, lat, accel.astype(np.float32), kappa.astype(np.float32)


def make_samples(n, seed):
    rng = np.random.default_rng(seed)
    recs = []
    for _ in range(n):
        lead_dist = rng.uniform(8, 80)
        light_red = bool(rng.random() < 0.25)
        lane_offset = rng.uniform(-1.2, 1.2)
        turn_cmd = int(rng.choice([0, 0, 0, -1, 1]))
        lane_change_cmd = 0 if turn_cmd != 0 else int(rng.choice([0, 0, 0, -1, 1]))
        v0 = float(rng.uniform(4.0, 16.0))
        imgs = render_rgb_scene(lead_dist, light_red, lane_offset, turn_cmd, lane_change_cmd, rng)
        lon, lat, accel, kappa = expert_driver(lead_dist, light_red, lane_offset,
                                               turn_cmd, lane_change_cmd, v0)
        act = np.stack([(accel - ACCEL_MEAN) / ACCEL_STD,
                        (kappa - CURVATURE_MEAN) / CURVATURE_STD], axis=-1)
        recs.append(dict(images=imgs, v0=v0, action=act.astype(np.float32),
                         lon=LON_DECISIONS.index(lon), lat=LAT_DECISIONS.index(lat),
                         lead_dist=lead_dist, light_red=light_red))
    return recs


N_TRAIN, N_TEST = 512, 96
train_recs = make_samples(N_TRAIN, seed=1)
test_recs = make_samples(N_TEST, seed=2)
print(f"train samples: {len(train_recs)}   test samples: {len(test_recs)}")

fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for r, rec in enumerate([train_recs[0], next(x for x in train_recs if x["light_red"])]):
    for c in range(3):
        axes[r, c].imshow(rec["images"][c]); axes[r, c].axis("off")
        axes[r, c].set_title(f"{CAM_NAMES[c]}\n{LON_DECISIONS[rec['lon']]}", fontsize=7)
plt.tight_layout(); plt.show()

assert len(train_recs[0]["images"]) == 3
assert train_recs[0]["action"].shape == (N_WAYPOINTS, 2)
print("\nCHECKPOINT 3 OK: synthetic multi-camera driving scenes built")


train samples: 512   test samples: 96

CHECKPOINT 3 OK: synthetic multi-camera driving scenes built


## 4. 実 VLM バックボーンのロード

`Qwen/Qwen3-VL-2B-Instruct` を実際にダウンロードして T4 に載せる。
Alpamayo 2 Super の `vlm_config.model_type` と同じ `qwen3_vl` 系統である。


In [5]:
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"

t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID)
try:
    vlm = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, dtype=DTYPE, attn_implementation=ATTN_IMPL, device_map=DEVICE)
except TypeError:
    # transformers v4 系では dtype ではなく torch_dtype
    vlm = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, torch_dtype=DTYPE, attn_implementation=ATTN_IMPL, device_map=DEVICE)
vlm.eval()
for p in vlm.parameters():
    p.requires_grad_(False)

load_s = time.time() - t0
n_params = sum(p.numel() for p in vlm.parameters())
vram = torch.cuda.memory_allocated() / 1024**3

print(f"model        : {MODEL_ID}")
print(f"class        : {type(vlm).__name__}")
print(f"config type  : {vlm.config.model_type}")
print(f"parameters   : {n_params:,}")
print(f"dtype        : {next(vlm.parameters()).dtype}")
print(f"attn impl    : {ATTN_IMPL}")
print(f"load time    : {load_s:.1f} s")
print(f"VRAM (weights): {vram:.2f} GiB / {total_vram:.1f} GiB")

assert n_params > 1.5e9, f"想定より小さいモデルがロードされた: {n_params:,}"
assert vram < 10.0, f"重みだけで VRAM を使いすぎ: {vram:.2f} GiB"
assert "qwen3_vl" in vlm.config.model_type, (
    f"Alpamayo2-Super と同系統 (qwen3_vl) であることを確認したい: {vlm.config.model_type}")
print("\nCHECKPOINT 4 OK: real Qwen3-VL backbone loaded on GPU "
      f"({n_params/1e9:.2f}B params, same family as Alpamayo2-Super)")


model        : Qwen/Qwen3-VL-2B-Instruct
class        : Qwen3VLForConditionalGeneration
config type  : qwen3_vl
parameters   : 2,127,532,032
dtype        : torch.float16
attn impl    : sdpa
load time    : 23.5 s
VRAM (weights): 3.96 GiB / 14.6 GiB

CHECKPOINT 4 OK: real Qwen3-VL backbone loaded on GPU (2.13B params, same family as Alpamayo2-Super)


## 5. Chain-of-Causation 形式の推論テキストを実 VLM に生成させる

Alpamayo の CoC は「閉集合の driving decision + 因果要因」という構造を持つ。
ここでは同じ構造をプロンプトで指定し、**実際にモデルにシーンを見せて**生成させる。

検証したいのは形式ではなく **grounding**、つまり「出力がシーンの内容に依存して変わるか」である。
形式だけを見る検査は簡単に騙される（プロンプトのテンプレートを複写しても通ってしまう）ので、

- 出力が構造化されパースできるか
- **信号の色をシーンから正しく読めているか**
- **decision がシーンによって変化するか（常に同じ答えを返していないか）**

の 3 つを測る。結果は良し悪しにかかわらずそのまま報告する。


In [6]:
# プレースホルダを <...> で書くとモデルがそれをそのまま複写することがあるため、
# 記法は使わず、期待する語彙のみを列挙する。
COC_PROMPT = (
    "You are the driving policy of an autonomous vehicle. Three camera images are given, "
    "in this order: the forward camera, the left cross camera, the right cross camera.\n"
    "In the forward camera, a traffic light is drawn as a small colored rectangle at the top "
    "centre of the image, and a lead vehicle, if present, is drawn as a red box on the road.\n\n"
    "Answer with exactly these four lines and nothing else. "
    "Do not repeat the instructions and do not use angle brackets.\n\n"
    "LIGHT: red or green\n"
    "LONGITUDINAL: one of these exact phrases: " + "; ".join(LON_DECISIONS) + "\n"
    "LATERAL: one of these exact phrases: " + "; ".join(LAT_DECISIONS) + "\n"
    "BECAUSE: one sentence linking what you see to the decision"
)


@torch.no_grad()
def generate_coc(rec, max_new_tokens=96):
    messages = [{"role": "user", "content":
                 [{"type": "image", "image": im} for im in rec["images"]]
                 + [{"type": "text", "text":
                     f"Ego speed is {rec['v0']:.1f} m/s.\n\n" + COC_PROMPT}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(DEVICE)
    out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = out[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(gen, skip_special_tokens=True)[0].strip()


import re

N_COC_EVAL = 24


def parse_coc(txt):
    """生成テキストから LIGHT / LONGITUDINAL / LATERAL を抜き出す."""
    up = txt.upper()
    light = None
    m = re.search(r"LIGHT\s*:\s*([A-Z]+)", up)
    if m:
        if "RED" in m.group(1):
            light = True
        elif "GREEN" in m.group(1):
            light = False
    lon = next((i for i, d in enumerate(LON_DECISIONS)
                if re.search(r"LONGITUDINAL\s*:.*" + re.escape(d.upper()), up)), None)
    lat = next((i for i, d in enumerate(LAT_DECISIONS)
                if re.search(r"LATERAL\s*:.*" + re.escape(d.upper()), up)), None)
    return light, lon, lat


t0 = time.time()
coc_texts = [generate_coc(rec) for rec in test_recs[:N_COC_EVAL]]
print(f"generation time: {time.time() - t0:.1f} s for {N_COC_EVAL} samples\n")

for i in range(3):
    rec = test_recs[i]
    print(f"--- sample {i}  (GT: {LON_DECISIONS[rec['lon']]} / {LAT_DECISIONS[rec['lat']]}, "
          f"lead={rec['lead_dist']:.0f}m, red_light={rec['light_red']}) ---")
    print(coc_texts[i])
    print()

parsed = [parse_coc(t) for t in coc_texts]
gt = test_recs[:N_COC_EVAL]

n_parse = sum(p[1] is not None and p[2] is not None for p in parsed)
light_pairs = [(p[0], r["light_red"]) for p, r in zip(parsed, gt) if p[0] is not None]
light_acc = (sum(a == b for a, b in light_pairs) / len(light_pairs)) if light_pairs else float("nan")
lon_pairs = [(p[1], r["lon"]) for p, r in zip(parsed, gt) if p[1] is not None]
lon_acc_zs = (sum(a == b for a, b in lon_pairs) / len(lon_pairs)) if lon_pairs else 0.0

uniq_lon = {p[1] for p in parsed if p[1] is not None}
uniq_lat = {p[2] for p in parsed if p[2] is not None}
majority_lon = max((sum(p[1] == u for p in parsed) for u in uniq_lon), default=0) / max(len(lon_pairs), 1)

# GT 側のクラス比率 (常に多数派を答えるだけで取れる精度 = チャンスレート)
gt_major = max(sum(r["lon"] == k for r in gt) for k in range(len(LON_DECISIONS))) / len(gt)

# reasoning テキストと decision が矛盾していないか (論文 5.3.2 の r_consistency に相当する検査)
STOP_IDX = LON_DECISIONS.index("Stop for static constraints")
says_stop = [bool(re.search(r"\bmust stop\b|\bshould stop\b|\bstop\b", t.split("BECAUSE")[-1], re.I))
             for t in coc_texts]
contradictions = [(sp, p[1]) for sp, p in zip(says_stop, parsed)
                  if sp and p[1] is not None and p[1] != STOP_IDX]
n_says_stop = sum(1 for sp, p in zip(says_stop, parsed) if sp and p[1] is not None)
contra_rate = (len(contradictions) / n_says_stop) if n_says_stop else float("nan")

print(f"パースできた割合                    : {n_parse}/{N_COC_EVAL}")
print(f"信号色 (red/green) の正答率         : {light_acc:.3f}   (チャンス 0.5)")
print(f"longitudinal decision の正答率      : {lon_acc_zs:.3f}   "
      f"(常に多数派を答えるだけで {gt_major:.3f})")
print(f"出力された longitudinal の種類数    : {len(uniq_lon)} / {len(LON_DECISIONS)}")
print(f"出力された lateral の種類数         : {len(uniq_lat)} / {len(LAT_DECISIONS)}")
print(f"最頻の longitudinal が占める割合    : {majority_lon:.3f}")
print(f"\nBECAUSE 節が「止まるべき」と書いた件数        : {n_says_stop}")
print(f"うち LONGITUDINAL が Stop 以外だった割合      : {contra_rate:.3f}"
      "   <- reasoning-action inconsistency")

# 検証するのは「形式に従えたか」だけ。精度はここでは主張しない (下で正直に評価する)。
assert all(len(t) > 20 for t in coc_texts), "VLM が空に近い出力しか返していない"
assert n_parse >= N_COC_EVAL * 0.7, f"パースできた出力が少なすぎる: {n_parse}/{N_COC_EVAL}"
assert not any("<" in t and ">" in t for t in coc_texts[:5]), (
    "モデルがプロンプトのプレースホルダ記法を複写している (形式指定を見直すこと)")

GROUNDED = (len(uniq_lon) > 1) and (lon_acc_zs > gt_major + 0.05)
print(f"""
CHECKPOINT 5 OK: real VLM produced parseable Chain-of-Causation traces
  形式には従えた。ただし grounding (シーンに応じて答えが変わるか) は別問題である。
  この 2B モデルの zero-shot decision は grounded か: {GROUNDED}
""")
if not GROUNDED:
    print(f"""[FINDING] 知覚は成立しているが、decision に接続していない。

  信号の色は {light_acc:.1%} の精度で読めている（チャンス 50%）。つまり「見えていない」のではない。
  それでも longitudinal decision は {len(uniq_lon)} 種類しか出力されず、
  多数派を答えるだけのベースライン ({gt_major:.3f}) を上回れていない ({lon_acc_zs:.3f})。

  さらに、BECAUSE 節で「赤信号なので止まらなければならない」と書きながら
  LONGITUDINAL には別の decision を選ぶ、という自己矛盾が {contra_rate:.1%} の割合で起きている。
  これは論文 Sec.5.3.2 が r_consistency 報酬で潰そうとしている現象そのものである
  —— 「流暢だが行動に反映されない推論」(fluent but causally disconnected explanations)。

  これは失敗ではなく、Alpamayo の設計理由を裏付ける結果である。
  NVIDIA は CoC を「プロンプトで出させる」のではなく、700K セグメントの構造化アノテーションを
  人手 + 自動ラベラで作り、SFT でモデルに焼き込み、さらに GRPO で reasoning-action consistency を
  報酬にして矯正している。プロンプトだけで済むならその全工程は不要だったはずである。""")
else:
    print("[FINDING] zero-shot でもシーンに応じて decision が変化している。")


generation time: 78.0 s for 24 samples

--- sample 0  (GT: Set speed tracking / Lane keeping & centering, lead=27m, red_light=False) ---
LIGHT: green
LONGITUDINAL: Set speed tracking
LATERAL: Lane keeping & centering
BECAUSE: The traffic light is green, so the vehicle can continue at its current speed.

--- sample 1  (GT: Stop for static constraints / Turn (intersection), lead=65m, red_light=True) ---
LIGHT: red  
LONGITUDINAL: Set speed tracking  
LATERAL: Lane keeping & centering  
BECAUSE: The traffic light is red, so the vehicle must stop, and the forward camera shows a lead vehicle, which is not a significant obstacle for lane keeping.

--- sample 2  (GT: Set speed tracking / Turn (intersection), lead=70m, red_light=False) ---
LIGHT: green
LONGITUDINAL: Set speed tracking
LATERAL: Lane keeping & centering
BECAUSE: The traffic light is green, so the vehicle can continue at its current speed, and the forward camera shows a clear lane and no immediate obstacles requiring a lane chang

## 6. VLM prefill の hidden state を取り出してキャッシュする

Alpamayo の要は「**expert が VLM を再実行せず、prefill 済みの `past_key_values` に attend する**」ことである。
ここでも同じ構造を取る：VLM の forward は 1 サンプルにつき 1 回だけ走らせて hidden state を保存し、
以降の expert 学習ではそれを使い回す（VLM は完全に凍結）。

fp16 の数値不安定に備え、NaN/Inf チェックを入れる。


In [7]:
N_COND_TOKENS = 32     # expert が attend する条件トークン数 (系列長を固定するためプールする)


@torch.no_grad()
def encode_scene(rec):
    """VLM prefill を 1 回だけ実行し、条件トークン (N_COND_TOKENS, H) を返す."""
    messages = [{"role": "user", "content":
                 [{"type": "image", "image": im} for im in rec["images"]]
                 + [{"type": "text", "text":
                     f"Ego speed is {rec['v0']:.1f} m/s.\n\n" + COC_PROMPT}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(DEVICE)
    out = vlm(**inputs, output_hidden_states=True, use_cache=False)
    h = out.hidden_states[-1][0]                       # (seq, hidden)
    # 系列長を N_COND_TOKENS に固定 (実物では KV キャッシュ全体に attend する)
    h = F.adaptive_avg_pool1d(h.transpose(0, 1).float().unsqueeze(0), N_COND_TOKENS)
    return h.squeeze(0).transpose(0, 1)                # (N_COND_TOKENS, hidden)


t0 = time.time()
probe = encode_scene(train_recs[0])
HIDDEN = probe.shape[-1]
print(f"VLM hidden size          : {HIDDEN}")
print(f"conditioning tokens      : {tuple(probe.shape)}")
print(f"per-sample encode time   : {time.time() - t0:.2f} s")

assert torch.isfinite(probe).all(), (
    "hidden state に NaN/Inf がある。fp16 で数値が壊れている可能性が高い")
print(f"hidden state stats       : mean={probe.mean():.4f} std={probe.std():.4f} "
      f"absmax={probe.abs().max():.2f}")


def encode_all(recs, tag):
    feats, v0s, acts, lons, lats = [], [], [], [], []
    t0 = time.time()
    for i, rec in enumerate(recs):
        feats.append(encode_scene(rec).cpu())
        v0s.append(rec["v0"]); acts.append(rec["action"])
        lons.append(rec["lon"]); lats.append(rec["lat"])
        if (i + 1) % 64 == 0:
            print(f"  [{tag}] {i+1}/{len(recs)}  ({time.time()-t0:.0f}s)")
    return (torch.stack(feats), torch.tensor(v0s), torch.from_numpy(np.stack(acts)),
            torch.tensor(lons), torch.tensor(lats))


Htr, Vtr, Atr, LONtr, LATtr = encode_all(train_recs, "train")
Hte, Vte, Ate, LONte, LATte = encode_all(test_recs, "test")

peak_vram = torch.cuda.max_memory_allocated() / 1024**3
print(f"\ncached train features: {tuple(Htr.shape)}")
print(f"cached test  features: {tuple(Hte.shape)}")
print(f"peak VRAM so far     : {peak_vram:.2f} GiB / {total_vram:.1f} GiB")

assert torch.isfinite(Htr).all() and torch.isfinite(Hte).all(), "キャッシュした特徴に NaN/Inf がある"
assert peak_vram < total_vram * 0.85, f"VRAM が上限に近すぎる: {peak_vram:.2f} GiB"

# 特徴がシーンの違いを本当に捉えているか (別シーン同士の距離 > 同一シーンの分散)
d_between = torch.cdist(Htr.mean(1), Htr.mean(1)).mean()
print(f"feature の平均ペア距離: {d_between:.3f}  (0 なら全シーンで同じ特徴 = 使い物にならない)")
assert d_between > 1e-3, "VLM 特徴がシーン間で区別されていない"
print("\nCHECKPOINT 6 OK: VLM prefill features cached, finite, and scene-discriminative")


VLM hidden size          : 2048
conditioning tokens      : (32, 2048)
per-sample encode time   : 0.17 s
hidden state stats       : mean=-0.0136 std=2.4578 absmax=101.57
  [train] 64/512  (12s)
  [train] 128/512  (25s)
  [train] 192/512  (37s)
  [train] 256/512  (49s)
  [train] 320/512  (61s)
  [train] 384/512  (73s)
  [train] 448/512  (84s)
  [train] 512/512  (96s)
  [test] 64/96  (12s)

cached train features: (512, 32, 2048)
cached test  features: (96, 32, 2048)
peak VRAM so far     : 4.14 GiB / 14.6 GiB
feature の平均ペア距離: 5.988  (0 なら全シーンで同じ特徴 = 使い物にならない)

CHECKPOINT 6 OK: VLM prefill features cached, finite, and scene-discriminative


## 7. Action expert を実 VLM の hidden state を条件に GPU 上で学習する

expert 側は公式実装の移植そのものである。VLM は凍結し、expert だけを学習する。
実物の Stage 1 が「KV キャッシュに stop-gradient をかけて expert を学習する」段階に相当する。


In [8]:
D_MODEL = 256


class ActionExpert(nn.Module):
    """VLM hidden state を条件に flow matching のベクトル場を予測する (公式 ExpertModel 相当)."""

    def __init__(self, cond_dim, d_model=D_MODEL, n_layers=4, n_heads=8):
        super().__init__()
        self.cond_proj = nn.Sequential(nn.LayerNorm(cond_dim), nn.Linear(cond_dim, d_model))
        self.action_in_proj = PerWaypointActionInProjV2(
            in_dims=list(action_space.get_action_space_dims()), out_dim=d_model, hidden_size=256)
        layer = nn.TransformerDecoderLayer(d_model, n_heads, d_model * 4, batch_first=True,
                                           norm_first=True, dropout=0.0, activation="gelu")
        # self-attn は非因果 (expert_non_causal_attention=True 相当)、cross-attn で VLM を見る
        self.expert = nn.TransformerDecoder(layer, n_layers, norm=nn.LayerNorm(d_model))
        self.wp_pos = nn.Parameter(torch.randn(1, N_WAYPOINTS, d_model) * 0.02)
        self.action_out_proj = nn.Linear(d_model, 2)
        # CoC の閉集合 decision ヘッド (reasoning-action consistency の評価に使う)
        self.lon_head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, len(LON_DECISIONS)))
        self.lat_head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, len(LAT_DECISIONS)))

    def encode_cond(self, cond):
        return self.cond_proj(cond)

    def decisions(self, memory):
        pooled = memory.mean(1)
        return self.lon_head(pooled), self.lat_head(pooled)

    def forward(self, noisy_action, timesteps, memory):
        emb = self.action_in_proj(noisy_action, timesteps) + self.wp_pos
        return self.action_out_proj(self.expert(tgt=emb, memory=memory))


expert = ActionExpert(cond_dim=HIDDEN).to(DEVICE)
n_expert = sum(p.numel() for p in expert.parameters())
print(f"action expert params : {n_expert:,}")
print(f"VLM params (frozen)  : {n_params:,}")
print(f"trainable ratio      : {n_expert / (n_expert + n_params):.4%}")

Htr_d, Vtr_d, Atr_d = Htr.to(DEVICE), Vtr.to(DEVICE), Atr.to(DEVICE)
Hte_d, Vte_d, Ate_d = Hte.to(DEVICE), Vte.to(DEVICE), Ate.to(DEVICE)
LONtr_d, LATtr_d = LONtr.to(DEVICE), LATtr.to(DEVICE)
LONte_d, LATte_d = LONte.to(DEVICE), LATte.to(DEVICE)

opt = torch.optim.AdamW(expert.parameters(), lr=2e-3, weight_decay=1e-4)
EPOCHS, BATCH = 250, 32
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

hist = {"flow": [], "test_flow": [], "coc": []}
t0 = time.time()
for ep in range(EPOCHS):
    expert.train()
    perm = torch.randperm(len(Htr_d), device=DEVICE)
    ep_flow, ep_coc, nb = 0.0, 0.0, 0
    for i in range(0, len(Htr_d), BATCH):
        idx = perm[i:i + BATCH]
        mem = expert.encode_cond(Htr_d[idx])
        td = diffusion.construct_training_data(Atr_d[idx])
        pred = expert(td["noisy_x"], td["timesteps"], mem)
        loss_flow = FlowMatching.compute_loss_from_pred(td, pred)
        lon_lg, lat_lg = expert.decisions(mem)
        loss_coc = F.cross_entropy(lon_lg, LONtr_d[idx]) + F.cross_entropy(lat_lg, LATtr_d[idx])
        loss = loss_flow + 0.3 * loss_coc
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(expert.parameters(), 1.0)
        opt.step()
        ep_flow += loss_flow.item(); ep_coc += loss_coc.item(); nb += 1
    sched.step()

    expert.eval()
    with torch.no_grad():
        mem_te = expert.encode_cond(Hte_d)
        td_te = diffusion.construct_training_data(Ate_d)
        test_flow = FlowMatching.compute_loss_from_pred(
            td_te, expert(td_te["noisy_x"], td_te["timesteps"], mem_te)).item()
    hist["flow"].append(ep_flow / nb); hist["coc"].append(ep_coc / nb)
    hist["test_flow"].append(test_flow)
    if ep % 20 == 0 or ep == EPOCHS - 1:
        print(f"epoch {ep:3d}  flow={ep_flow/nb:.4f}  coc={ep_coc/nb:.4f}  test_flow={test_flow:.4f}")

train_s = time.time() - t0
peak_vram = torch.cuda.max_memory_allocated() / 1024**3
print(f"\ntraining time : {train_s:.1f} s")
print(f"peak VRAM     : {peak_vram:.2f} GiB / {total_vram:.1f} GiB")
print(f"flow loss     : {hist['flow'][0]:.4f} -> {hist['flow'][-1]:.4f} "
      f"({(1 - hist['flow'][-1]/hist['flow'][0])*100:.1f}% 減少)")

assert hist["flow"][-1] < hist["flow"][0] * 0.4, "flow matching 損失が十分に下がっていない"
assert hist["test_flow"][-1] < hist["test_flow"][0] * 0.5, "test 損失が下がっていない (過学習/未収束)"
assert peak_vram < total_vram * 0.85, f"VRAM が上限に近すぎる: {peak_vram:.2f} GiB"
print("\nCHECKPOINT 7 OK: action expert trained on real VLM features, losses converged")


action expert params : 5,042,696
VLM params (frozen)  : 2,127,532,032
trainable ratio      : 0.2365%
epoch   0  flow=4.4942  coc=2.9341  test_flow=3.0680
epoch  20  flow=1.5792  coc=0.5963  test_flow=1.6294
epoch  40  flow=1.4014  coc=0.4928  test_flow=1.5632
epoch  60  flow=1.3017  coc=0.1710  test_flow=1.5677
epoch  80  flow=1.2632  coc=0.1210  test_flow=1.6266
epoch 100  flow=1.2010  coc=0.0167  test_flow=1.6125
epoch 120  flow=1.1861  coc=0.0043  test_flow=1.6627
epoch 140  flow=1.1557  coc=0.0024  test_flow=1.5522
epoch 160  flow=1.0913  coc=0.0016  test_flow=1.5164
epoch 180  flow=1.0032  coc=0.0012  test_flow=1.4938
epoch 200  flow=0.9929  coc=0.0010  test_flow=1.4423
epoch 220  flow=0.9866  coc=0.0009  test_flow=1.4487
epoch 240  flow=0.9906  coc=0.0009  test_flow=1.4626
epoch 249  flow=0.9791  coc=0.0009  test_flow=1.4510

training time : 131.3 s
peak VRAM     : 4.45 GiB / 14.6 GiB
flow loss     : 4.4942 -> 0.9791 (78.2% 減少)

CHECKPOINT 7 OK: action expert trained on real VLM 

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(hist["flow"], label="train"); axes[0].plot(hist["test_flow"], label="test")
axes[0].set_yscale("log"); axes[0].set_xlabel("epoch"); axes[0].set_title("flow matching loss")
axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(hist["coc"], color="darkorange"); axes[1].set_xlabel("epoch")
axes[1].set_title("Chain-of-Causation decision loss"); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


## 8. 推論 — 10 ステップの Euler 積分と minADE 評価

公式デフォルトと同じ `inference_step=10` でサンプリングし、minADE で評価する。


In [10]:
@torch.no_grad()
def predict(H, num_samples=6, inference_step=NUM_INFERENCE_STEPS):
    expert.eval()
    mem = expert.encode_cond(H)                       # VLM は再実行しない
    lon_lg, lat_lg = expert.decisions(mem)
    mem_rep = mem.repeat_interleave(num_samples, dim=0)

    def step_fn(x, t):
        return expert(x, t, mem_rep)

    act = diffusion.sample(H.shape[0] * num_samples, step_fn, DEVICE, inference_step)
    return act.view(H.shape[0], num_samples, N_WAYPOINTS, 2), lon_lg, lat_lg


def min_ade(pred_act, gt_act, v0):
    b, n = pred_act.shape[0], pred_act.shape[1]
    xy_p, _, _ = action_space.action_to_traj(
        pred_act.reshape(b * n, N_WAYPOINTS, 2), v0.repeat_interleave(n))
    xy_p = xy_p.reshape(b, n, N_WAYPOINTS, 2)
    xy_g, _, _ = action_space.action_to_traj(gt_act, v0)
    return (xy_p - xy_g.unsqueeze(1)).norm(dim=-1).mean(-1).min(dim=1).values


t0 = time.time()
pred_act, lon_lg, lat_lg = predict(Hte_d, num_samples=6)
infer_s = (time.time() - t0) / len(Hte_d)

ade6 = min_ade(pred_act, Ate_d, Vte_d)
lon_acc = (lon_lg.argmax(-1) == LONte_d).float().mean().item()
lat_acc = (lat_lg.argmax(-1) == LATte_d).float().mean().item()
in_bounds = action_space.is_within_bounds(pred_act.reshape(-1, N_WAYPOINTS, 2)).float().mean().item()

# ベースライン: 学習セットの平均 action を常に出す (条件付けを一切使わない)
mean_act = Atr_d.mean(0, keepdim=True).expand(len(Hte_d), -1, -1).unsqueeze(1)
ade_base = min_ade(mean_act, Ate_d, Vte_d)

print(f"minADE_6 @6.4s (learned)            : {ade6.mean():.4f} m (median {ade6.median():.4f} m)")
print(f"minADE_6 @6.4s (mean-action ベース) : {ade_base.mean():.4f} m")
print(f"ベースラインに対する改善             : {(1 - ade6.mean()/ade_base.mean())*100:.1f}%")
print(f"CoC longitudinal decision accuracy  : {lon_acc:.3f}   (zero-shot VLM: 前節参照)")
print(f"CoC lateral      decision accuracy  : {lat_acc:.3f}")
print(f"action が物理境界内の割合            : {in_bounds:.3f}")
print(f"expert 推論時間 (6 samples/scene)    : {infer_s*1000:.1f} ms/scene")

assert torch.isfinite(pred_act).all(), "予測 action に NaN/Inf がある"
# 固定閾値ではなく「条件付けを使わないベースラインを明確に上回るか」で判定する
assert ade6.mean() < ade_base.mean() * 0.6, (
    f"VLM 条件付けが効いていない: learned={ade6.mean():.3f} m vs baseline={ade_base.mean():.3f} m")
assert lon_acc > 0.6, f"longitudinal decision accuracy が低い: {lon_acc:.3f}"
assert lat_acc > 0.6, f"lateral decision accuracy が低い: {lat_acc:.3f}"
assert in_bounds > 0.95, f"物理境界を外れる action が多い: {in_bounds:.3f}"
print("\nCHECKPOINT 8 OK: expert beats the unconditioned baseline -> VLM features carry signal")


minADE_6 @6.4s (learned)            : 4.1389 m (median 1.6411 m)
minADE_6 @6.4s (mean-action ベース) : 11.2882 m
ベースラインに対する改善             : 63.3%
CoC longitudinal decision accuracy  : 0.969   (zero-shot VLM: 前節参照)
CoC lateral      decision accuracy  : 0.927
action が物理境界内の割合            : 1.000
expert 推論時間 (6 samples/scene)    : 4.7 ms/scene

CHECKPOINT 8 OK: expert beats the unconditioned baseline -> VLM features carry signal


In [11]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, lat_id in zip(axes, range(len(LAT_DECISIONS))):
    hits = (LATte_d == lat_id).nonzero()
    if len(hits) == 0:
        ax.axis("off"); continue
    idx = int(hits[0])
    xy_g, _, _ = action_space.action_to_traj(Ate_d[idx:idx+1], Vte_d[idx:idx+1])
    xy_p, _, _ = action_space.action_to_traj(pred_act[idx], Vte_d[idx:idx+1].repeat(6))
    for s in range(6):
        ax.plot(xy_p[s, :, 0].cpu(), xy_p[s, :, 1].cpu(), color="tab:blue", alpha=.35, lw=1.2,
                label="predicted (6 samples)" if s == 0 else None)
    ax.plot(xy_g[0, :, 0].cpu(), xy_g[0, :, 1].cpu(), "k--", lw=2.2, label="GT")
    ax.set_title(f"GT: {LAT_DECISIONS[lat_id]}\npred: {LAT_DECISIONS[lat_lg[idx].argmax().item()]}",
                 fontsize=9)
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.axis("equal")
    ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()


## 9. 推論ステップ数のトレードオフ（論文 Table 12 の追試）

論文は $\delta_t=0.2$（5 ステップ）でも劣化は無視できると述べている。実 GPU 上で確認する。


In [12]:
step_counts = [1, 2, 3, 5, 10, 20]
res = []
for s in step_counts:
    pa, _, _ = predict(Hte_d, num_samples=6, inference_step=s)
    res.append(min_ade(pa, Ate_d, Vte_d).mean().item())

base = res[step_counts.index(10)]
for s, r in zip(step_counts, res):
    print(f"steps={s:3d}  minADE_6 = {r:.4f} m  ({(r/base-1)*100:+.1f}% vs 10 steps)")

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(step_counts, res, marker="o")
ax.axvline(NUM_INFERENCE_STEPS, color="r", ls="--", lw=1, label="official default (10)")
ax.set_xscale("log"); ax.set_xlabel("Euler steps"); ax.set_ylabel("minADE_6 [m]")
ax.set_title("inference steps vs accuracy (Tesla T4)"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

deg5 = res[step_counts.index(5)] / base - 1
deg1 = res[step_counts.index(1)] / base - 1
spread = (max(res) - min(res)) / base

print(f"\n5 ステップでの劣化 : {deg5*100:+.1f}%   (論文: 無視できる程度)")
print(f"1 ステップでの劣化 : {deg1*100:+.1f}%")
print(f"全体のばらつき幅   : {spread*100:.1f}%")

# 論文が主張しているのは「5 ステップでも 10 ステップと大差ない」ことだけなので、それだけを検証する。
assert abs(deg5) < 0.30, f"5 ステップの劣化が論文の主張より大きい: {deg5*100:+.1f}%"
assert all(np.isfinite(r) for r in res), "minADE に NaN/Inf がある"
print("\nCHECKPOINT 9 OK: 5 steps is within 30% of 10 steps, as the paper claims")

if deg1 < 0.05:
    print(f"""
[FINDING] このモデルでは 1 ステップですら 10 ステップと同等である ({deg1*100:+.1f}%)。
  論文の設定とは異なる。理由は、この expert が学習したベクトル場がほぼ線形で、
  条件付き平均に向かって真っ直ぐ流れているだけだからである。
  Euler 1 ステップでもその平均に着いてしまうので、ステップ数を増やす意味がない。

  実物の Alpamayo で複数ステップが必要なのは、行動分布が本質的に多峰だからである
  (交差点で「止まる / 先に行く」のように、平均を取るとどちらでもない軌道になる場面)。
  多峰性を表現できるだけの容量が expert に無いと、このように flow が単なる回帰に退化する。
  ステップ数の ablation は「モデルが多峰性を捉えているか」の診断にも使えるということでもある。""")


steps=  1  minADE_6 = 4.0526 m  (-6.8% vs 10 steps)
steps=  2  minADE_6 = 4.0719 m  (-6.3% vs 10 steps)
steps=  3  minADE_6 = 4.1913 m  (-3.6% vs 10 steps)
steps=  5  minADE_6 = 4.1364 m  (-4.8% vs 10 steps)
steps= 10  minADE_6 = 4.3465 m  (+0.0% vs 10 steps)
steps= 20  minADE_6 = 4.2277 m  (-2.7% vs 10 steps)

5 ステップでの劣化 : -4.8%   (論文: 無視できる程度)
1 ステップでの劣化 : -6.8%
全体のばらつき幅   : 6.8%

CHECKPOINT 9 OK: 5 steps is within 30% of 10 steps, as the paper claims

[FINDING] このモデルでは 1 ステップですら 10 ステップと同等である (-6.8%)。
  論文の設定とは異なる。理由は、この expert が学習したベクトル場がほぼ線形で、
  条件付き平均に向かって真っ直ぐ流れているだけだからである。
  Euler 1 ステップでもその平均に着いてしまうので、ステップ数を増やす意味がない。

  実物の Alpamayo で複数ステップが必要なのは、行動分布が本質的に多峰だからである
  (交差点で「止まる / 先に行く」のように、平均を取るとどちらでもない軌道になる場面)。
  多峰性を表現できるだけの容量が expert に無いと、このように flow が単なる回帰に退化する。
  ステップ数の ablation は「モデルが多峰性を捉えているか」の診断にも使えるということでもある。


## 10. 保存したチェックポイントが再ロードして同じ結果を出すか

「学習できた」だけでなく「保存した成果物が再現する」ことまで確認する。


In [13]:
CKPT = "/kaggle/working/alpamayo2_action_expert.pt"
torch.save({"state_dict": expert.state_dict(), "cond_dim": HIDDEN, "d_model": D_MODEL,
            "model_id": MODEL_ID, "n_waypoints": N_WAYPOINTS, "dt": DT}, CKPT)
size_mb = os.path.getsize(CKPT) / 1024**2
print(f"saved: {CKPT}  ({size_mb:.1f} MB)")

expert2 = ActionExpert(cond_dim=HIDDEN).to(DEVICE)
ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)
expert2.load_state_dict(ck["state_dict"])
expert2.eval()

# 同じノイズ・同じ条件で両者のベクトル場が一致するか
torch.manual_seed(1234)
x_probe = torch.randn(8, N_WAYPOINTS, 2, device=DEVICE)
t_probe = torch.full((8, 1, 1), 0.5, device=DEVICE)
with torch.no_grad():
    mem = expert.encode_cond(Hte_d[:8])
    mem2 = expert2.encode_cond(Hte_d[:8])
    v1 = expert(x_probe, t_probe, mem)
    v2 = expert2(x_probe, t_probe, mem2)

max_diff = (v1 - v2).abs().max().item()
print(f"reload 後のベクトル場の最大差分: {max_diff:.3e}")
assert max_diff < 1e-5, f"再ロードしたモデルの出力が一致しない: {max_diff:.3e}"

# 保存したモデルで実際に軌道を出せるか
expert_backup, expert = expert, expert2
pa2, _, _ = predict(Hte_d, num_samples=6)
ade2 = min_ade(pa2, Ate_d, Vte_d).mean().item()
expert = expert_backup
print(f"reload したモデルの minADE_6: {ade2:.4f} m (元: {ade6.mean():.4f} m)")
assert abs(ade2 - ade6.mean().item()) < 0.5, "再ロード後に性能が大きく変わっている"
print("\nCHECKPOINT 10 OK: checkpoint round-trips and still produces valid trajectories")


saved: /kaggle/working/alpamayo2_action_expert.pt  (19.3 MB)
reload 後のベクトル場の最大差分: 0.000e+00
reload したモデルの minADE_6: 4.1127 m (元: 4.1389 m)

CHECKPOINT 10 OK: checkpoint round-trips and still produces valid trajectories


## 11. 本番 GPU へ移すときに何が変わるか


In [14]:
print("=" * 78)
print("ALL CHECKPOINTS PASSED")
print("=" * 78)
print(f"GPU                    : {gpu_name} (sm_{cap[0]}{cap[1]})")
print(f"VLM backbone           : {MODEL_ID}  ({n_params/1e9:.2f}B, {vlm.config.model_type})")
print(f"dtype / attention      : {DTYPE} / {ATTN_IMPL}")
print(f"action expert          : {n_expert:,} params (trained here)")
print(f"peak VRAM              : {torch.cuda.max_memory_allocated()/1024**3:.2f} GiB "
      f"/ {total_vram:.1f} GiB")
print(f"minADE_6 @6.4s         : {ade6.mean():.4f} m "
      f"(unconditioned baseline {ade_base.mean():.4f} m)")
print(f"CoC decision accuracy  : lon {lon_acc:.3f} / lat {lat_acc:.3f} (trained head)")
print(f"zero-shot VLM grounded : {GROUNDED}   "
      f"(lon acc {lon_acc_zs:.3f} vs majority-class {gt_major:.3f})")
print()
print("-" * 78)
print("本番 GPU (A100 / H100) へ移すときに変わる点")
print("-" * 78)
print("""
1. dtype            : fp16 -> bf16。sm_80 以上なら BF16_OK が True になり自動で切り替わる。
                      判定は torch.cuda.is_bf16_supported() ではなく get_device_capability()。

2. attention        : sdpa -> flash_attention_2。Turing (sm_75) では flash-attn 2 が
                      ビルドすらできないため、本 notebook は sdpa に落としている。

3. バックボーン     : Qwen3-VL-2B -> nvidia/Alpamayo2-Super (34B)。
                      公式実測で VLM 側 67 GiB / expert 側 71 GiB を要するため 80GB 級 x2 が要る。
                      Alpamayo-1.5-10B なら 24GB 以上の単一 GPU で動く。

4. バージョンピン   : 公式は torch==2.8.0 / transformers==4.57.1 にハードピンし、
                      flash-attn>=2.8.3 をソースビルドする。Python は 3.12 系。
                      公開チェックポイントの config.json は transformers_version 4.57.6 で
                      記録されており、ここにずれがあることに注意。

5. 条件付けの経路   : 本 notebook は hidden_states を平均プールして 32 トークンに固定したが、
                      公式は VLM prefill の past_key_values 全体に expert が層ごとに attend する
                      (expert は VLM と同じ 64 層で、hidden_size だけ 5120 -> 1536 に細くしてある)。

6. データ           : 合成シーン -> nvidia/PhysicalAI-Autonomous-Vehicles (gated, 要ライセンス同意)。
                      7 カメラ環状 x 4 フレーム、1080x1920 の実映像になる。

7. 学習規模         : expert のみ 120 epoch -> Stage1 (80,000 時間) / Stage2 SFT (700K CoC) /
                      Stage3 GRPO の 3 段階。RL には報酬モデルと closed-loop シミュレータが要る。
""")


ALL CHECKPOINTS PASSED
GPU                    : Tesla T4 (sm_75)
VLM backbone           : Qwen/Qwen3-VL-2B-Instruct  (2.13B, qwen3_vl)
dtype / attention      : torch.float16 / sdpa
action expert          : 5,042,696 params (trained here)
peak VRAM              : 4.69 GiB / 14.6 GiB
minADE_6 @6.4s         : 4.1389 m (unconditioned baseline 11.2882 m)
CoC decision accuracy  : lon 0.969 / lat 0.927 (trained head)
zero-shot VLM grounded : False   (lon acc 0.500 vs majority-class 0.500)

------------------------------------------------------------------------------
本番 GPU (A100 / H100) へ移すときに変わる点
------------------------------------------------------------------------------

1. dtype            : fp16 -> bf16。sm_80 以上なら BF16_OK が True になり自動で切り替わる。
                      判定は torch.cuda.is_bf16_supported() ではなく get_device_capability()。

2. attention        : sdpa -> flash_attention_2。Turing (sm_75) では flash-attn 2 が
                      ビルドすらできないため、本 notebook は sdpa に落としている。

3. バックボーン     : 

## Takeaways

- **Alpamayo 2 Super そのものは T4 に載らない**（34B / 実測 67 GiB）。ただしバックボーンは `qwen3_vl` 系統なので、
  同系統の 2B を代わりに置けば**同じパイプライン構造をそのまま実 GPU で回せる**。
- **汎用 VLM に形式を指示しても CoC は得られない。** 2B モデルは信号の色を高精度で読めているのに、
  decision はほぼ一定値に張り付き、多数派ベースラインを超えなかった（CHECKPOINT 5 参照）。
  さらに「赤信号だから止まるべき」と書きながら別の decision を選ぶ自己矛盾が観測された。
  **問題は知覚ではなく、知覚と行動の接続である。**
  これは Alpamayo が 700K の構造化アノテーション + SFT + GRPO の consistency 報酬を積んでいる理由そのものである。
- **形式だけを検査する assert は、この失敗を素通りさせる。** 最初の版では「LONGITUDINAL と LATERAL を含むか」
  「閉集合の語彙が現れるか」しか見ておらず、モデルがプロンプトのテンプレートを複写しても緑になっていた。
  検証すべきは形式ではなく、**答えが入力によって変わるか**である。
- **action expert 側は代用ではなく本物の移植**である。公式の `PerWaypointActionInProjV2` /
  `FlowMatching` / `UnicycleAccelCurvatureActionSpace` を、公式 `config.json` の実数値で動かし、
  実 VLM の hidden state を条件に GPU 上で学習させて収束することを確認した。
- **T4 で踏む罠は 2 つ**。`torch.cuda.is_bf16_supported()` は Turing でも `True` を返すので
  dtype 判定に使ってはいけない。flash-attn 2 は sm_80 未満でビルドできないので `sdpa` に落とす必要がある。
- **VLM は 1 サンプルにつき 1 回しか走らない**。prefill 済みの特徴をキャッシュして expert が使い回す構造は
  Alpamayo そのもので、これが 10 ステップの拡散を回してもレイテンシが伸びない理由である。

理論の全体像は [alpamayo2.md](alpamayo2.md)、公式実装の数式レベルの解剖は
[alpamayo2.ipynb](alpamayo2.ipynb)（CPU 版）を参照。
